# LoRA × CLIP — Vision-Language Retrieval on a Colab GPU

This notebook extends the repo's from-scratch LoRA implementation to a **vision-language model**. The original Microsoft LoRA paper did not report multimodal experiments, so this is an extension rather than a paper reproduction.

**Task in this notebook**
- Model: `openai/clip-vit-base-patch32`
- Task: image-text retrieval
- Dataset: `lambdalabs/pokemon-blip-captions`
- LoRA targets: attention `q_proj` and `v_proj` in the CLIP text + vision towers
- Metrics: image-to-text and text-to-image `Recall@1 / Recall@5`

**Why this setup?**
- It is a real VLM task.
- It fits easily on a T4 / L4 GPU.
- It uses the same LoRA primitives already used elsewhere in this repo.

**Runtime budget**
- Default smoke test: roughly 10–25 minutes on a Colab T4.
- Stronger runs: increase `max_samples` or switch to a larger image-text dataset after the pipeline works end-to-end.


In [ ]:
!nvidia-smi


## 1. Load the project code

Pick **one** option below.

**Option A (recommended):** zip the repo's `code/` folder and upload it.
```bash
# on your laptop, from inside project_LoRA/:
zip -r code.zip code -x 'code/external/*' -x '*/__pycache__/*' -x '*/.venv/*'
```

**Option B:** clone the public repo mirror if available.


In [ ]:
# --- Option A: upload code.zip ---
from google.colab import files
import os, zipfile

uploaded = files.upload()  # drag-drop code.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')

def find_code_root(base='/content', max_depth=2):
    for dirpath, _, filenames in os.walk(base):
        depth = dirpath[len(base):].count(os.sep)
        if depth > max_depth:
            continue
        if 'requirements.txt' in filenames and os.path.isdir(os.path.join(dirpath, 'lora')):
            return dirpath
    return None

root = find_code_root()
assert root is not None, "Couldn't find a folder with requirements.txt + lora/ in the uploaded zip."
os.chdir(root)
print('Working dir:', os.getcwd())
print(os.listdir(root))


In [ ]:
# --- Option B: clone from a public GitHub mirror (uncomment if applicable) ---
# !git clone https://github.com/austinlu2005/myLoRA.git /content/myLoRA
# %cd /content/myLoRA/code


In [ ]:
!pip install -q -r requirements.txt


## 2. Imports


In [ ]:
import sys, os, json, math, time
from pathlib import Path

sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoProcessor

from dataloaders.vlm import load_clip_retrieval_data, make_clip_collate_fn
from evaluation.vlm_metrics import compute_clip_retrieval_metrics
from lora.save_load import load_lora_state_dict
from models.vlm_wrapper import build_clip_lora
from training.optim import build_optimizer, build_scheduler
from training.vlm_trainer import CLIPRetrievalTrainer
from utils.param_utils import print_trainable_parameters
from utils.seed import set_seed

print(
    'torch:', torch.__version__,
    '| cuda:', torch.cuda.is_available(),
    '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
)


## 3. Hyperparameters

These defaults are intentionally sized for a notebook run on T4 / L4. The dataset is small and acts as a clean multimodal smoke test for the LoRA extension.


In [ ]:
CFG = {
    'model_name': 'openai/clip-vit-base-patch32',
    'tower': 'both',
    'dataset': 'lambdalabs/pokemon-blip-captions',
    'split': 'train',
    'max_samples': 800,
    'eval_ratio': 0.2,
    'rank': 8,
    'alpha': 16,
    'dropout': 0.1,
    'batch_size': 16,
    'num_epochs': 8,
    'lr': 1e-4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.06,
    'grad_clip': 1.0,
    'seed': 42,
    'log_steps': 10,
    'max_text_length': 77,
    'primary_metric': 'mean_recall@1',
    'recall_at': (1, 5),
    'output_dir': '/content/runs_vlm_clip',
}
CFG


## 4. Load the Dataset and Preview Examples


In [ ]:
set_seed(CFG['seed'])

data = load_clip_retrieval_data(
    dataset_name=CFG['dataset'],
    split=CFG['split'],
    max_samples=CFG['max_samples'],
    eval_ratio=CFG['eval_ratio'],
    seed=CFG['seed'],
)
train_raw = data['train']
eval_raw = data['validation']
image_col = data['image_col']
text_col = data['text_col']

print(f"train={len(train_raw)} | eval={len(eval_raw)}")
print(f"image column: {image_col} | text column: {text_col}")

def to_text(value):
    if isinstance(value, list):
        return value[0] if value else ''
    return str(value)

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for idx, ax in enumerate(axes.flat):
    ex = train_raw[idx]
    ax.imshow(ex[image_col].convert('RGB'))
    ax.set_title(to_text(ex[text_col])[:70])
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5. Build CLIP + LoRA

This uses the repo's shared `inject_lora(...)` path. For CLIP we target attention `q_proj` and `v_proj` modules in the vision and text transformers.


In [ ]:
processor = AutoProcessor.from_pretrained(CFG['model_name'])
collate_fn = make_clip_collate_fn(
    processor,
    image_col=image_col,
    text_col=text_col,
    max_length=CFG['max_text_length'],
)

model, replaced = build_clip_lora(
    model_name=CFG['model_name'],
    rank=CFG['rank'],
    alpha=CFG['alpha'],
    dropout=CFG['dropout'],
    tower=CFG['tower'],
)
print(f'Injected LoRA into {len(replaced)} modules')
print(replaced[:8])
print_trainable_parameters(model)


## 6. Train


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
steps_per_epoch = math.ceil(len(train_raw) / CFG['batch_size'])
num_training_steps = steps_per_epoch * CFG['num_epochs']

optimizer = build_optimizer(model, lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = build_scheduler(
    optimizer,
    num_training_steps=num_training_steps,
    warmup_ratio=CFG['warmup_ratio'],
)

trainer = CLIPRetrievalTrainer(
    model=model,
    train_dataset=train_raw,
    eval_dataset=eval_raw,
    collate_fn=collate_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    batch_size=CFG['batch_size'],
    num_epochs=CFG['num_epochs'],
    device=device,
    grad_clip=CFG['grad_clip'],
    log_steps=CFG['log_steps'],
    output_dir=CFG['output_dir'],
    primary_metric=CFG['primary_metric'],
    ks=CFG['recall_at'],
)

t0 = time.time()
history = trainer.train()
wall = time.time() - t0

result = {
    'task': 'image_text_retrieval',
    'dataset': CFG['dataset'],
    'primary_metric': CFG['primary_metric'],
    'best_metric': trainer.best_metric,
    'wall_seconds': wall,
    'history': history,
    'config': CFG,
    'replaced_modules': replaced,
}
os.makedirs(CFG['output_dir'], exist_ok=True)
with open(os.path.join(CFG['output_dir'], 'result.json'), 'w') as f:
    json.dump(result, f, indent=2)

print(f"best {CFG['primary_metric']}: {trainer.best_metric:.4f}")
print(f"wall minutes: {wall / 60:.1f}")
print(json.dumps(history[-1], indent=2))


## 7. Plot the Learning Curve


In [ ]:
epochs = [row['epoch'] for row in history]
losses = [row['eval_loss'] for row in history]
mr1 = [row.get('mean_recall@1') for row in history]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(epochs, losses, marker='o')
axes[0].set_title('Eval Loss')
axes[0].set_xlabel('epoch')

axes[1].plot(epochs, mr1, marker='o')
axes[1].set_title('Mean Recall@1')
axes[1].set_xlabel('epoch')

plt.tight_layout()
plt.show()


## 8. Reload the Best LoRA Checkpoint and Show Retrieval


In [ ]:
best_model, _ = build_clip_lora(
    model_name=CFG['model_name'],
    rank=CFG['rank'],
    alpha=CFG['alpha'],
    dropout=CFG['dropout'],
    tower=CFG['tower'],
)
load_lora_state_dict(best_model, os.path.join(CFG['output_dir'], 'lora_best.pt'))
best_model = best_model.to(device).eval()

def encode_dataset(model, dataset, batch_size=32):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    image_embeds, text_embeds = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            out = model(**batch)
        image_embeds.append(out.image_embeds.cpu())
        text_embeds.append(out.text_embeds.cpu())
    return torch.cat(image_embeds), torch.cat(text_embeds)

eval_image_embeds, eval_text_embeds = encode_dataset(best_model, eval_raw)
metrics = compute_clip_retrieval_metrics(eval_image_embeds, eval_text_embeds, ks=CFG['recall_at'])
print(metrics)

query_idx = 0
scores = eval_image_embeds[query_idx] @ eval_text_embeds.T
topk = scores.topk(5).indices.tolist()

query = eval_raw[query_idx]
plt.figure(figsize=(4, 4))
plt.imshow(query[image_col].convert('RGB'))
plt.title('query image')
plt.axis('off')
plt.show()

print('ground truth:', to_text(query[text_col]))
print('\nTop retrieved captions:')
for rank, idx in enumerate(topk, start=1):
    print(f'{rank}. {to_text(eval_raw[idx][text_col])}')


## Notes

- This is a **LoRA extension to a VLM**, not a replication of the original paper.
- The default dataset is intentionally small so the notebook is practical on free or low-cost Colab hardware.
- To scale up after the smoke test works, swap in a larger image-text dataset and keep the same `build_clip_lora(...)` code path.
- The saved checkpoint is `lora_best.pt`, consistent with the rest of the repo.
